In [6]:
import json
import requests
import os
import subprocess
from urllib.parse import urlparse
import os.path

class AccessibilityChecker:
    def __init__(self, json_file, download_dir="downloads"):
        self.metadata_repository = self.load_metadata(json_file)
        self.download_dir = download_dir
        os.makedirs(self.download_dir, exist_ok=True)

    def load_metadata(self, json_file):
        with open(json_file, 'r') as file:
            data = json.load(file)
        return {artifact_id: artifact['uri'] for artifact_id, artifact in data['artifacts'].items()}

    def is_git_repo(self, uri):
        return any(uri.startswith(prefix) for prefix in [
            "https://github.com/", "http://github.com/",
            "https://gitlab.com/", "http://gitlab.com/"
        ])

    def clone_repo(self, artifact_id, uri):
        repo_name = os.path.splitext(os.path.basename(urlparse(uri).path))[0]
        repo_path = os.path.join(self.download_dir, repo_name)
        try:
            subprocess.run(["git", "clone", uri, repo_path], check=True)
            print(f"\n🚀 Artifact Accessible: Git repository detected.")
            print(f"📦 Artifact ID: {artifact_id}")
            print(f"🔗 Repo URL: {uri}")
            print(f"📁 Cloned to: {repo_path}")
            print("✨✨✨ GIT REPO READY FOR EXPLORATION ✨✨✨\n")
        except subprocess.CalledProcessError:
            print(f"❌ Failed to clone repository {uri}")

    def download_artifact(self, artifact_id, uri):
        try:
            response = requests.get(uri, allow_redirects=True, timeout=10)
            if response.status_code == 200:
                filename = self.infer_filename(uri, artifact_id)
                file_path = os.path.join(self.download_dir, filename)
                with open(file_path, "wb") as file:
                    file.write(response.content)
                print(f"\n✅ Artifact Accessible: '{artifact_id}'")
                print(f"🔗 Download URL: {uri}")
                print(f"📄 Saved as: {filename}")
                print(f"📍 Location: {file_path}")
                print("🎉🎉🎉 DOWNLOAD COMPLETE 🎉🎉🎉\n")
                return response.content
        except requests.RequestException:
            pass
        return None

    def infer_filename(self, uri, artifact_id):
        parsed_url = urlparse(uri)
        name = os.path.basename(parsed_url.path)
        return name if name else f"{artifact_id}.bin"

    def check_accessibility(self):
        for artifact_id, uri in self.metadata_repository.items():
            print(f"\n🔍 Checking artifact: {artifact_id}")
            if self.is_git_repo(uri):
                self.clone_repo(artifact_id, uri)
            else:
                artifact_data = self.download_artifact(artifact_id, uri)
                if artifact_data:
                    if len(artifact_data) < 500:
                        print(f"🔎 Preview of '{artifact_id}':")
                        print(artifact_data.decode(errors='ignore')[:200])
                else:
                    print(f"❌ Artifact '{artifact_id}' is NOT accessible.\n")

if __name__ == "__main__":
    json_file_path = 'artifacts.json'
    checker = AccessibilityChecker(json_file_path)
    checker.check_accessibility()



🔍 Checking artifact: artifact_1

🚀 Artifact Accessible: Git repository detected.
📦 Artifact ID: artifact_1
🔗 Repo URL: https://github.com/hihey54/hicss58
📁 Cloned to: downloads\hicss58
✨✨✨ GIT REPO READY FOR EXPLORATION ✨✨✨


🔍 Checking artifact: artifact_2
❌ Failed to clone repository https://github.com/githubkilroy/TCN-Dataset

🔍 Checking artifact: artifact_3

🚀 Artifact Accessible: Git repository detected.
📦 Artifact ID: artifact_3
🔗 Repo URL: https://github.com/TheAlgorithms/Python
📁 Cloned to: downloads\Python
✨✨✨ GIT REPO READY FOR EXPLORATION ✨✨✨

